# RagSolver — resultados de evaluación

Lee `data/eval/results/*.json` y `data/eval/runs/<run_id>/summary.json` (generados por `docs/reproduce.md` §3-4) y arma las tablas del informe. Ejecutar desde la raíz del repo.

In [1]:
import json, glob
from pathlib import Path
import pandas as pd

R = Path('../data/eval/results') if Path('../data').exists() else Path('data/eval/results')
load = lambda n: json.loads((R / n).read_text(encoding='utf-8')) if (R / n).exists() else None

## 1. Retriever sobre Mathlib (MathlibQR, LeanSearch v2)

Tabla de muestra: gold de MathlibQR/MPR + premisas de miniF2F + 3000 distractores (`docs/DECISIONS.md` #31-32 explica por qué los números de la Fase 1 original estaban inflados).

In [2]:
full = load('fase1_redo_mathlibqr_sample3903_norerank.json')
s100 = load('fase1_redo_mathlibqr_sample3903_100q.json')
full_idx = load('fase1_full_mathlibqr_100q.json')  # índice completo (278k), scripts/run_full_index.sh
rows = []
for name, res in (('946 consultas, sin rerank', full), ('100 al azar, con rerank', s100), ('100 al azar, índice completo', full_idx)):
    if res:
        for v in ('denso', 'bm25', 'hibrido', 'hibrido_rerank'):
            if v in res:
                rows.append({'corrida': name, 'variante': v, 'cobertura': res['coverage_ratio'], **{m: round(res[v][m], 3) for m in ('recall@10', 'ndcg@10', 'mrr')}, 'recall@10 condicionado': round(res[v]['recall@10'] / res['coverage_ratio'], 3)})
pd.DataFrame(rows)

,corrida,variante,cobertura,recall@10,ndcg@10,mrr,recall@10 condicionado
0,"946 consultas, sin rerank",denso,0.838,0.662,0.568,0.542,0.790
1,"946 consultas, sin rerank",bm25,0.838,0.595,0.481,0.449,0.710
2,"946 consultas, sin rerank",hibrido,0.838,0.655,0.543,0.512,0.782
3,"100 al azar, con rerank",denso,0.860,0.760,0.631,0.591,0.884
4,"100 al azar, con rerank",bm25,0.860,0.560,0.437,0.403,0.651
5,"100 al azar, con rerank",hibrido,0.860,0.660,0.533,0.501,0.767
6,"100 al azar, con rerank",hibrido_rerank,0.860,0.710,0.617,0.592,0.826
7,"100 al azar, índice completo",denso,0.860,0.410,0.297,0.268,0.477
8,"100 al azar, índice completo",bm25,0.860,0.250,0.196,0.184,0.291
9,"100 al azar, índice completo",hibrido,0.860,0.380,0.276,0.250,0.442


## 2. Premisas gold de miniF2F (pruebas verificadas de DeepSeek-Prover-V2)

Consulta = enunciado completo (informal o formal). En el pipeline la unidad de consulta es el subobjetivo del Planificador, así que esto es una cota inferior.

In [3]:
prem = load('fase5_premises.json')
prem_full = load('fase5_premises_full.json')
pd.DataFrame([{'indice': name, 'consulta': k.replace('query_', ''), **{m: round(v[m], 3) for m in v}} for name, res in (('muestra 3903', prem), ('completo', prem_full)) if res for k, v in res.items() if k.startswith('query_')])

,indice,consulta,recall@8,recall@20,ndcg@10,mrr
0,muestra 3903,nl,0.080,0.153,0.076,0.129
1,muestra 3903,formal,0.118,0.180,0.106,0.176
2,completo,nl,0.006,0.010,0.006,0.016
3,completo,formal,0.009,0.018,0.007,0.015


## 3. Gold informal (30 consultas sobre el PDF de ejemplo, 15 dependientes de figura)

In [4]:
inf = load('fase5_informal_gold.json')
if inf:
    display(pd.DataFrame([{k: v for k, v in inf.items() if k not in ('rows',)}]).T)
    display(pd.DataFrame(inf['rows']))

,0
n_queries,30
recall@5,1.0
recall@5_wilson95,"[0.886, 1.0]"
figure_dependent,"{'n': 15, 'recall@5': 1.0, 'figure_coverage_me..."
mrr,1.0


,query_id,hit@k,rank,top_items,figure_coverage
0,f01,True,1,"[8, 28, 17, 25, 22]",1.000000
1,f02,True,1,"[16, 55, 75, 93, 1]",1.000000
2,f03,True,1,"[17, 25, 28, 75, 1]",1.000000
3,f04,True,1,"[22, 1, 99, 57, 3]",1.000000
4,f05,True,1,"[24, 49, 8, 25, 36]",1.000000
5,f06,True,1,"[28, 8, 25, 72, 36]",0.666667
6,f07,True,1,"[37, 71, 26, 97, 29]",1.000000
7,f08,True,1,"[61, 42, 53, 75, 74]",1.000000
8,f09,True,1,"[68, 63, 25, 17, 1]",1.000000
9,f10,True,1,"[80, 79, 10, 90, 82]",1.000000


## 4. Contaminación corpus informal ↔ miniF2F-v2

In [5]:
cont = load('fase5_contamination.json')
{k: v for k, v in cont.items() if k != 'pairs_over_threshold'} if cont else 'sin resultados'

{'threshold': 0.9,
 'n_cards': 100,
 'n_minif2f': 488,
 'n_pairs_over_threshold': 0,
 'max_similarity': 0.8181180953979492,
 'top10_most_similar': [[23, 'imo_1959_p1', 0.8181180953979492],
  [58, 'induction_divisibility_3div2tooddnp1', 0.8117700815200806],
  [76, 'algebra_abpbcpcageq3_sumaonsqrtapbgeq3onsqrt2', 0.7985193729400635],
  [70, 'amc12a_2020_p4', 0.7692816853523254],
  [9, 'induction_sum_odd', 0.7569718360900879],
  [74, 'algebra_abpbcpcageq3_sumaonsqrtapbgeq3onsqrt2', 0.7562429308891296],
  [15, 'aime_1984_p1', 0.739531934261322],
  [62, 'mathd_numbertheory_109', 0.7309191226959229],
  [47, 'amc12a_2003_p23', 0.7208618521690369],
  [60, 'mathd_numbertheory_483', 0.7189329862594604]]}

## 5. Ablaciones end-to-end (propuesta §8.2-8.4)

In [6]:
runs = sorted(glob.glob(str(R.parent / 'runs' / '*' / 'summary.json')))
for s in runs:
    summ = json.loads(Path(s).read_text(encoding='utf-8'))
    print(Path(s).parent.name)
    display(pd.DataFrame(summ).T)

eval-v1


,n,pass@1,pass@1_wilson95,llm_calls_mean,tokens_mean,wall_s_median,retrieval_calls_mean,llm_calls_mean_ci95_bootstrap,tokens_mean_ci95_bootstrap,wall_s_mean_ci95_bootstrap,pass_by_attempt,lemma_use_precision,re_retrieval_success,error_taxonomy,self_assessed_correct
full,60,0.15,"[0.081, 0.261]",7.62,35190,287.3,8.03,"[6.8, 8.48]","[30849.37, 39932.1]","[278.01, 402.55]","{'1': 0.117, '2': 0.15, '3': 0.15}",0.002,0.0,"{'planning': 49, 'lean_syntax': 6, 'exhausted'...",None
no_rag,60,0.15,"[0.081, 0.261]",7.83,23360,152.8,0.0,"[7.02, 8.6]","[19730.08, 27196.38]","[149.79, 215.02]","{'1': 0.15, '2': 0.15, '3': 0.15}",None,None,"{'missing_lemma': 25, 'planning': 41, 'exhaust...",None
no_critic,60,0.067,"[0.026, 0.159]",3.28,16433,62.4,5.2,"[3.13, 3.43]","[14168.08, 18858.3]","[84.89, 127.28]",{'1': 0.067},0.001,None,"{'unknown': 44, 'exhausted': 44, 'llm_output_i...",None
baseline_a,60,0.083,"[0.036, 0.181]",1.13,3012,5.2,0.0,"[1.05, 1.22]","[2045.9, 4110.2]","[12.66, 27.52]",{'1': 0.083},None,None,"{'unknown': 47, 'llm_output_invalid:baseline_a...",None
baseline_b,60,0.0,"[0.0, 0.06]",1.05,2214,5.9,0.0,"[1.0, 1.1]","[1384.35, 3126.98]","[11.73, 27.87]",{'1': 0.0},None,None,{'llm_output_invalid:baseline_b': 3},55


f4-trivial-full


,n,pass@1,pass@1_wilson95,llm_calls_mean,tokens_mean,wall_s_median,lemma_use_precision,re_retrieval_success,error_taxonomy,self_assessed_correct
full,5,1.0,"[0.566, 1.0]",3.0,7679,83.5,0.022,None,{},None


f5-ablations


,n,pass@1,pass@1_wilson95,llm_calls_mean,tokens_mean,wall_s_median,lemma_use_precision,re_retrieval_success,error_taxonomy,self_assessed_correct
full,13,0.692,"[0.424, 0.873]",5.69,13740,8.6,0.011,0.0,"{'lean_syntax': 8, 'exhausted': 4, 'planning':...",None
no_rag,13,0.769,"[0.497, 0.918]",5.31,4709,5.6,None,None,"{'missing_lemma': 4, 'lean_syntax': 8, 'exhaus...",None
no_critic,13,0.462,"[0.232, 0.709]",3.0,7231,6.1,0.004,None,"{'unknown': 7, 'exhausted': 7}",None
baseline_a,13,0.615,"[0.355, 0.823]",1.0,410,1.1,None,None,{'unknown': 5},None
baseline_b,13,0.0,"[0.0, 0.228]",1.0,412,1.5,None,None,{},13


### Comparación pareada (McNemar) entre configuraciones sobre los mismos problemas

In [7]:
from itertools import combinations
from statistics import NormalDist
for s in runs:
    res = [json.loads(l) for l in open(Path(s).parent / 'results.jsonl', encoding='utf-8')]
    by = {}
    for r in res:
        by.setdefault(r['config'], {})[r['problem_id']] = r['status'] == 'done'
    for a, b in combinations(sorted(k for k in by if k != 'baseline_b'), 2):  # B no se verifica: su 'status' no es comparable
        common = set(by[a]) & set(by[b])
        n01 = sum(1 for p in common if by[a][p] and not by[b][p]); n10 = sum(1 for p in common if by[b][p] and not by[a][p])
        # McNemar exacto (binomial) para n pequeño
        from math import comb
        n = n01 + n10
        pval = min(1.0, 2 * sum(comb(n, k) for k in range(0, min(n01, n10) + 1)) / 2 ** n) if n else 1.0
        print(f'{Path(s).parent.name}: {a} vs {b}: solo-{a}={n01} solo-{b}={n10} p={pval:.3f}')

eval-v1: baseline_a vs full: solo-baseline_a=1 solo-full=5 p=0.219
eval-v1: baseline_a vs no_critic: solo-baseline_a=4 solo-no_critic=3 p=1.000
eval-v1: baseline_a vs no_rag: solo-baseline_a=0 solo-no_rag=4 p=0.125
eval-v1: full vs no_critic: solo-full=7 solo-no_critic=2 p=0.180
eval-v1: full vs no_rag: solo-full=2 solo-no_rag=2 p=1.000
eval-v1: no_critic vs no_rag: solo-no_critic=1 solo-no_rag=6 p=0.125
f5-ablations: baseline_a vs full: solo-baseline_a=0 solo-full=1 p=1.000
f5-ablations: baseline_a vs no_critic: solo-baseline_a=2 solo-no_critic=0 p=0.500
f5-ablations: baseline_a vs no_rag: solo-baseline_a=0 solo-no_rag=2 p=0.500
f5-ablations: full vs no_critic: solo-full=3 solo-no_critic=0 p=0.250
f5-ablations: full vs no_rag: solo-full=0 solo-no_rag=1 p=1.000
f5-ablations: no_critic vs no_rag: solo-no_critic=0 solo-no_rag=4 p=0.125
